# Shreve Week 01 — Probability Review & Stochastic Processes

**Week 1** — Probability Review & Stochastic Processes

This notebook teaches **probability review & stochastic processes** in the style of our graduate probability notebook: definitions from **Shreve**, intuition and examples from **Baxter & Rennie**, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve | Baxter & Rennie |
|------|-------|--------|-----------------|
| 1 | **Random variables & vectors** | Ch. 1.1–1.2 | Ch. 1.1–1.3 |
| 2 | **Expectation, variance, conditioning** | Ch. 1.3 | Ch. 2.1 |
| 3 | **Filtrations & adapted processes** | Ch. 2.1 | Ch. 2.2–2.3 |
| 4 | **Martingales (discrete preview)** | Ch. 2.3 | Ch. 2.4 |
| 5 | **Brownian motion as a random path** | Ch. 1.4, Ch. 3 preview | Ch. 3.1 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. **Shreve** (*Stochastic Calculus for Finance II*) — rigorous measure-theoretic treatment; see chapter pointers in each section.
5. **Baxter & Rennie** (*Financial Calculus*) — market intuition, replication, and worked examples; see spotlight sections.

**References:**
- **Shreve** Vol II — Ch. 1–2 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Baxter & Rennie**, *Financial Calculus* — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Random Variables and Random Vectors

A **random variable** $X$ is a measurable function $X: \Omega \to \mathbb{R}$. A **random vector** $(X_1,\ldots,X_d)$ assigns a vector to each outcome.

For continuous $X$ with PDF $f$:

$$E[X] = \int_{-\infty}^{\infty} x f(x)\,dx, \quad \text{Var}(X) = E[X^2] - (E[X])^2$$

**Shreve Ch. 1.1–1.2:** random variables, distributions, and joint distributions on $\mathbb{R}^d$.


In [ ]:
# Joint normal vector — marginals and correlation
rho = 0.6
cov = np.array([[1.0, rho], [rho, 1.0]])
rng = np.random.default_rng(0)
samples = rng.multivariate_normal([0, 0], cov, size=50_000)

print(f"E[X1] ≈ {samples[:, 0].mean():.4f}, E[X2] ≈ {samples[:, 1].mean():.4f}")
print(f"Var(X1) ≈ {samples[:, 0].var():.4f}, Var(X2) ≈ {samples[:, 1].var():.4f}")
print(f"Cov(X1,X2) ≈ {np.cov(samples, rowvar=False)[0, 1]:.4f} (theory ρ={rho})")


---
# Part 2 — Conditional Expectation

**Definition:** $E[X \mid \mathcal{G}]$ is the unique $\mathcal{G}$-measurable r.v. with $E[X \mathbf{1}_A] = E[E[X|\mathcal{G}] \mathbf{1}_A]$ for all $A \in \mathcal{G}$.

**Tower property:** $E[E[X \mid \mathcal{G}]] = E[X]$.

**Shreve Ch. 1.3:** conditional expectation as the best predictor in $L^2$.


In [ ]:
# Tower property: E[Y] = E[E[Y|X]]
rng = np.random.default_rng(1)
X = rng.uniform(0, 1, 100_000)
Y = rng.normal(X, 0.1)  # Y | X ~ N(X, 0.1)

E_Y = Y.mean()
E_E_Y_given_X = X.mean()  # E[Y|X] = X for this construction
print(f"E[Y]           = {E_Y:.4f}")
print(f"E[E[Y|X]] = E[X] = {E_E_Y_given_X:.4f}")


---
# Part 3 — Filtrations and Stochastic Processes

A **filtration** $\{\mathcal{F}_t\}_{t \geq 0}$ is an increasing family of $\sigma$-algebras: $\mathcal{F}_s \subseteq \mathcal{F}_t$ for $s \leq t$.

A **stochastic process** $\{X_t\}$ is **adapted** if $X_t$ is $\mathcal{F}_t$-measurable for each $t$.

**Shreve Ch. 2.1:** information grows over time; trading strategies must be adapted (you cannot use future prices).


In [ ]:
# Random walk S_n — natural filtration F_n = sigma(S_0,...,S_n)
n_steps = 200
rng = np.random.default_rng(2)
increments = rng.choice([-1, 1], size=n_steps)
S = np.concatenate([[0], np.cumsum(increments)])

fig, ax = plt.subplots()
ax.plot(S, lw=1.5)
ax.set_xlabel("time n")
ax.set_ylabel("S_n")
ax.set_title("Symmetric random walk (adapted to its own filtration)")
plt.show()


---
# Part 4 — Martingales (Preview)

$M_t$ is a **martingale** w.r.t. $\{\mathcal{F}_t\}$ if $E[|M_t|] < \infty$ and $E[M_t \mid \mathcal{F}_s] = M_s$ for $s \leq t$.

Fair games: expected future wealth equals current wealth given all history.

**Shreve Ch. 2.3:** martingale definition; symmetric random walk is a martingale.


In [ ]:
# Verify E[S_n] = 0 for symmetric random walk
rng = np.random.default_rng(3)
n_sims, n = 100_000, 50
walks = rng.choice([-1, 1], size=(n_sims, n))
S_n = walks.sum(axis=1)
print(f"E[S_{n}] simulated = {S_n.mean():.4f} (theory 0)")
print(f"Var(S_{n}) simulated = {S_n.var():.4f} (theory {n})")


---
# Part 5 — Paths in $C[0,T]$ (Brownian Preview)

Brownian motion $W_t$ lives on **path space** $C[0,T]$ — continuous functions on $[0,T]$.

**Shreve Ch. 1.4:** probability on $C[0,T]$; Ch. 3 constructs $W$ from random walks.


In [ ]:
# Scaled random walk → Brownian motion (Donsker preview)
def brownian_from_rw(T=1.0, n=500, n_paths=5, seed=4):
    rng = np.random.default_rng(seed)
    dt = T / n
    fig, ax = plt.subplots()
    for _ in range(n_paths):
        dW = rng.choice([-1, 1], size=n) / np.sqrt(n)
        W = np.concatenate([[0], np.cumsum(dW)])
        t = np.linspace(0, T, n + 1)
        ax.plot(t, W, alpha=0.8)
    ax.set_title(f"Scaled random walk → W_t (n={n})")
    ax.set_xlabel("t")
    ax.set_ylabel("W_t")
    plt.show()

brownian_from_rw(n=2000)


**Your turn:** Increase `n` in the scaled random walk. How does the path roughness change?


---
## Baxter & Rennie spotlight — The parable of the bookmaker

Before measure theory, Baxter & Rennie motivate pricing with a **bookmaker** who sets odds so no arbitrage exists. The same logic underlies derivative pricing:

- **Expectation pricing** (Ch. 1.1): fair price = expected payoff (needs a probability measure).
- **Arbitrage pricing** (Ch. 1.2): fair price = cost of a **replicating** strategy.

**Read:** *Financial Calculus*, Parable + Ch. 1.1–1.3. Compare Shreve's rigorous $(\Omega, \mathcal{F}, P)$ setup with B&R's market-first intuition.


**B&R Example (Ch. 2.1):** A one-step branch — stock $S$ goes to $S_0 u$ or $S_0 d$ with probabilities $p$ and \$1-p$. A claim paying $V_u$ or $V_d$ has **no-arbitrage price**

$$V_0 = e^{-r\Delta t}\big(p^* V_u + (1-p^*) V_d\big)$$

where $p^*$ is the **risk-neutral probability** (not necessarily physical $p$).


In [ ]:
# B&R Ch. 2.1 — one-step binomial risk-neutral price
S0, u, d, r, dt = 100, 1.1, 0.9, 0.05, 1/252
Vu, Vd = 10, 0  # e.g. digital or call-like payoff at step 1
p_star = (np.exp(r * dt) - d) / (u - d)
V0 = np.exp(-r * dt) * (p_star * Vu + (1 - p_star) * Vd)
print(f"Risk-neutral prob p* = {p_star:.4f}")
print(f"No-arbitrage claim price V0 = {V0:.4f}")


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. For $X \sim N(0,1)$, $Y = X^2$, find $E[Y]$ and $\text{Var}(Y)$.

2. Prove the tower property for discrete $\mathcal{G} = \sigma(X)$ using definition of conditional expectation.

3. Show that if $M_n$ is a martingale then $E[M_n] = E[M_0]$ for all $n$.

4. For a symmetric random walk $S_n$, verify $E[S_n^2] = n$ by induction.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** $E[Y] = E[X^2] = 1$. $E[Y^2] = E[X^4] = 3$, so $\text{Var}(Y) = 2$.

**2.** $E[E[X|\sigma(X)]] = E[X]$ by definition of conditional expectation with $A = \Omega$.

**3.** $E[M_n] = E[E[M_n|\mathcal{F}_0]] = E[M_0]$ by martingale property at $t=0$.

**4.** $E[S_{n+1}^2] = E[(S_n + D)^2] = E[S_n^2] + 2E[S_n D] + E[D^2] = n + 0 + 1 = n+1$.

</details>


---
## Further reading

### Shreve
- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 1–2 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

### Baxter & Rennie
- **Baxter & Rennie**, *Financial Calculus*, Parable + Ch. 1–2.2 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf)
  - Ch. 1.1–1.3: expectation vs arbitrage (complements Shreve Ch. 1–2).
  - Ch. 2.1–2.2: binomial branch/tree — discrete martingales before Brownian motion.
  - **Appendix 2** (notation) and **Appendix 4** (glossary) are useful references.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
